# Evaluation

Scores finished runs and makes the report's figures. Use a **GPU** runtime (validation and predictions need it). Run section 1 every session, then any later section on its own. Pick checkpoints and examples on validation only; the test subset is for final scores.

In [ ]:
REPO_URL = "https://github.com/davidyi0/handwritten-math-to-latex.git"
REPO_DIR = "/content/CoMER"
DRIVE_DATA = "/content/drive/MyDrive/hme100k"
LOCAL_DATA = "/content/hme100k"
RUNS = "/content/drive/MyDrive/comer_runs"
RUN = "baseline"  # baseline, imagenet, photo_aug, curriculum (config ours/configs/{RUN}.yaml)
TEST_IMAGES = "/content/test_subset"
PREDICTIONS = f"{RUNS}/predictions"
FIGURES = f"{RUNS}/figures"

## 1. Every session

In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from ours.colab_setup import setup
setup()

In [ ]:
!python -m ours.fetch_data --local {LOCAL_DATA} --drive {DRIVE_DATA}
!python -m ours.expand_vocab
!mkdir -p {PREDICTIONS} {FIGURES}

import glob, re
# Best checkpoint of a run, by the val_ExpRate in its file name (validation subset, so no test data is used to pick it).
def best_checkpoint(run):
    files = glob.glob(f"{RUNS}/{run}/checkpoints/*val_ExpRate=*.ckpt")
    return max(files, key=lambda f: float(re.search(r"val_ExpRate=([0-9.]+)\.ckpt", f).group(1)))

CKPT = best_checkpoint(RUN)
print(RUN, "->", CKPT)

## 2. Full validation score (all 3,722 validation images)

In [ ]:
!python -m ours.finetune validate -c ours/configs/{RUN}.yaml --ckpt_path "{CKPT}" --run_dir /content/val_full/{RUN} --data.val_limit 0 2>&1 | tee {RUNS}/{RUN}/val_full_log.txt

## 3. Test subset predictions (final scores only)

Export the 2,000 test images once per session, then predict with the chosen run, and once with the zero-shot (expanded, not fine-tuned) checkpoint. Reruns resume.

In [ ]:
!python -m ours.test_splits --data {LOCAL_DATA} --export {TEST_IMAGES}

In [ ]:
!python -m ours.predict --ckpt "{CKPT}" --model {RUN} --images {TEST_IMAGES} --out {PREDICTIONS}/{RUN}_clean.jsonl

In [ ]:
!python -m ours.predict --ckpt checkpoints/crohme_expanded_hme100k.ckpt --model zero_shot --images {TEST_IMAGES} --out {PREDICTIONS}/zero_shot_clean.jsonl

## 4. Compilation validity

Install LaTeX once per session. The labels' own compile rate is the reference: it should be close to 100%.

In [ ]:
!apt-get -qq install texlive-latex-base texlive-latex-recommended texlive-fonts-recommended > /dev/null
!python -c "import csv,json; [print(json.dumps({'id':r['id'],'model':'labels','condition':'clean','label':r['label']})) for r in csv.DictReader(open('ours/hme100k_test_subset.tsv'),delimiter='\\t')]" > /content/labels.jsonl
!python -m ours.compile_check /content/labels.jsonl --field label --out {PREDICTIONS}/labels.compile.jsonl

In [ ]:
!python -m ours.compile_check {PREDICTIONS}/{RUN}_clean.jsonl

## 5. Attention maps (validation images; same seed gives the same images for every run)

In [ ]:
!python -m ours.attention_maps --ckpt "{CKPT}" --val {LOCAL_DATA} --count 12 --out {FIGURES}/attention/{RUN}

## 6. Figures

In [ ]:
runs = [r for r in ("baseline", "photo_aug", "curriculum", "imagenet") if glob.glob(f"{RUNS}/{r}/**/events.out.tfevents*", recursive=True)]
args = " ".join(f"{r}={RUNS}/{r}" for r in runs)
!python -m ours.plot_curves {args} --out {FIGURES}/training_curves.png

In [ ]:
!python -m ours.figure_datasets --val {LOCAL_DATA} --out {FIGURES}/figure_datasets.png

## Done: release the GPU

In [ ]:
from google.colab import runtime
runtime.unassign()